In [ ]:
# Example 1: Public API surface — verify the main classes are importable.
from cloudpathlib import CloudPath, S3Path, GSPath, AzureBlobPath
from cloudpathlib import S3Client, GSClient, AzureBlobClient
from cloudpathlib import implementation_registry

print(f"CloudPath: {CloudPath}")
print(f"S3Path: {S3Path}")
print(f"GSPath: {GSPath}")
print(f"AzureBlobPath: {AzureBlobPath}")
print(f"Registered implementations: {sorted(implementation_registry.keys())}")

assert 's3' in implementation_registry
assert 'gs' in implementation_registry
assert 'azure' in implementation_registry


In [ ]:
# Example 2: CloudPath URI dispatch — construct without authenticating.
from cloudpathlib import CloudPath, S3Path, GSPath, AzureBlobPath

s3_p = CloudPath("s3://mybucket/folder/file.txt")
gs_p = CloudPath("gs://mybucket/folder/file.txt")
az_p = CloudPath("az://mycontainer/folder/file.txt")

assert isinstance(s3_p, S3Path)
assert isinstance(gs_p, GSPath)
assert isinstance(az_p, AzureBlobPath)

print(f"S3:    {s3_p} -> bucket={s3_p.bucket}, key={s3_p.key}")
print(f"GS:    {gs_p} -> bucket={gs_p.bucket}, blob={gs_p.blob}")
print(f"Azure: {az_p} -> container={az_p.container}, blob={az_p.blob}")

assert s3_p.bucket == "mybucket"
assert s3_p.key == "folder/file.txt"


In [ ]:
# Example 3: Path arithmetic — pathlib-like operations that don't touch a backend.
from cloudpathlib import CloudPath

p = CloudPath("s3://bucket/a/b/c/file.tar.gz")

print(f"name:   {p.name}")
print(f"stem:   {p.stem}")
print(f"suffix: {p.suffix}")
print(f"suffixes: {p.suffixes}")
print(f"parent: {p.parent}")
print(f"parents: {list(p.parents)}")
print(f"parts:  {p.parts}")

child = p.parent / "sibling.json"
assert str(child) == "s3://bucket/a/b/c/sibling.json"
assert p.name == "file.tar.gz"
assert p.suffix == ".gz"
assert p.stem == "file.tar"
assert p.with_suffix(".zip").name == "file.tar.zip"


In [ ]:
# Example 4: Local mock client — full read/write/list round-trip with no network.
from cloudpathlib.local import LocalS3Client, LocalS3Path

client = LocalS3Client()
root = client.CloudPath("s3://test-bucket")

(root / "hello.txt").write_text("hello world")
(root / "sub/nested.txt").write_text("nested content")
(root / "sub/other.bin").write_bytes(b"\x00\x01\x02")

assert (root / "hello.txt").read_text() == "hello world"
assert (root / "sub/other.bin").read_bytes() == b"\x00\x01\x02"
assert (root / "hello.txt").exists()
assert not (root / "missing.txt").exists()

all_files = sorted(str(p.relative_to(root)) for p in root.rglob("*") if p.is_file())
print(f"Files under {root}: {all_files}")
assert "hello.txt" in all_files

# Note: Client has no close() in cloudpathlib (0.24); the local backend's scratch
# dir is process-temp and needs no explicit teardown.


In [ ]:
# Example 5: Local mock client — copy, delete, glob patterns.
from cloudpathlib.local import LocalGSClient

client = LocalGSClient()
root = client.CloudPath("gs://demo-bucket")
(root / "a.txt").write_text("A")
(root / "b.txt").write_text("B")
(root / "c.log").write_text("C")

txt_files = sorted(p.name for p in root.glob("*.txt"))
assert txt_files == ["a.txt", "b.txt"], txt_files

dst = root / "copied/a.txt"
(root / "a.txt").copy(dst)
assert dst.exists() and dst.read_text() == "A"

(root / "b.txt").unlink()
assert not (root / "b.txt").exists()

remaining = sorted(p.name for p in root.rglob("*") if p.is_file())
print(f"Remaining files: {remaining}")
assert "a.txt" in remaining and "c.log" in remaining
# Note: Client has no close() in cloudpathlib (0.24) — nothing to tear down.
print("cloudpathlib smoke tests passed")
